# General Analysis: Attention Weights vs. Norm-Based Maps

Python 3 version of `General_Analysis.ipynb` (Sections 3 and 6 of
[Clark et al., 2019](https://arxiv.org/abs/1906.04341)) that runs every analysis
on the attention weights $\alpha$ **and** on the norm-based maps
$\|\alpha f(x)\|$ of [Kobayashi et al., 2020](https://www.aclweb.org/anthology/2020.emnlp-main.574/).

Data: token-level maps written by `extract_norms.py`, e.g.
```
python preprocess_unlabeled.py --data-file $DATA/unlabeled.txt --bert-dir bert-base-uncased
python extract_norms.py --preprocessed-data-file $DATA/unlabeled.json --bert-dir bert-base-uncased \
    --outputs attns,norms,fx_norms
python head_distances.py --attn-data-file $DATA/unlabeled_norms.pkl --key attns --outfile $DATA/head_distances_attns.pkl
python head_distances.py --attn-data-file $DATA/unlabeled_norms.pkl --key norms --outfile $DATA/head_distances_norms.pkl
```

**Scale.** Rows of $\alpha$ sum to one; rows of $\|\alpha f(x)\|$ do not. Each
statistic is therefore reported for the norms both *raw* (the quantity plotted by
Kobayashi et al.) and *row-normalized* (the share of a token's summed norm going to
each position, on the same scale as $\alpha$ and usable with Clark et al.'s
thresholds). Entropies and JS divergences are only defined for normalized rows.

In [ ]:
import os
import sys

import numpy as np
import seaborn as sns
import sklearn.manifold
from matplotlib import cm
from matplotlib import pyplot as plt

import analysis_utils as au
import utils

sns.set_style("darkgrid")

DATA_DIR = os.environ.get("ATTN_DATA_DIR", "./data")
DATA_FILE = os.path.join(DATA_DIR, "unlabeled_norms.pkl")
HEAD_DISTANCES = {k: os.path.join(DATA_DIR, "head_distances_%s.pkl" % k)
                  for k in ["attns", "norms"]}

In [ ]:
# list of dicts with "tokens", "attns" [layers, heads, n, n],
# "norms" [layers, heads, n, n] and optionally "fx_norms" [layers, heads, n]
data = utils.load_pickle(DATA_FILE)
n_layers, n_heads = data[0]["attns"].shape[:2]
print(len(data), "examples;", n_layers, "layers x", n_heads, "heads")

### Average attention to particular tokens/positions (Sections 3.1 and 3.2)

In [ ]:
# (map name, key, normalize)
MAPS = [("attention", "attns", False),
        ("norm (raw)", "norms", False),
        ("norm (normalized)", "norms", True)]
avg_stats = {name: au.token_stats(data, key, normalize)
             for name, key, normalize in MAPS}

In [ ]:
BLACK, GREEN, SEA, BLUE = "k", "#59d98e", "#159d82", "#3498db"
PURPLE, GREY, RED, ORANGE = "#9b59b6", "#95a5a6", "#e74c3c", "#f39c12"


def get_data_points(head_data):
  xs, ys, avgs = [], [], []
  for layer in range(head_data.shape[0]):
    for head in range(head_data.shape[1]):
      ys.append(head_data[layer, head])
      xs.append(1 + layer)
    avgs.append(head_data[layer].mean())
  return xs, ys, avgs


def add_line(stats, key, ax, color, label, plot_avgs=True):
  xs, ys, avgs = get_data_points(stats[key])
  ax.scatter(xs, ys, s=12, label=label, color=color)
  if plot_avgs:
    ax.plot(1 + np.arange(len(avgs)), avgs, color=color)
  ax.legend(loc="best")
  ax.set_xlabel("Layer")


# Figure 2 of Clark et al. (average attention per head), one column per map
fig, axes = plt.subplots(3, len(MAPS), figsize=(5 * len(MAPS), 10))
for col, (name, _, _) in enumerate(MAPS):
  stats = avg_stats[name]
  for key, color, label in [("cls", RED, "[CLS]"), ("sep", BLUE, "[SEP]"),
                            ("punct", PURPLE, ". or ,")]:
    add_line(stats, key, axes[0, col], color, label)
  for key, color, label in [("rest_sep", BLUE, "other -> [SEP]"),
                            ("sep_sep", GREEN, "[SEP] -> [SEP]")]:
    add_line(stats, key, axes[1, col], color, label)
  # "right" = attention from token i to i+1 (np.eye(n, n, 1)). NOTE: the
  # original notebook labels "left" as "next token" and "right" as "prev
  # token", which contradicts its own clustering code and the selectors.
  for key, color, label in [("right", RED, "next token"),
                            ("left", BLUE, "prev token"),
                            ("self", PURPLE, "current token")]:
    add_line(stats, key, axes[2, col], color, label, plot_avgs=False)
  axes[0, col].set_title(name)
  for row in range(3):
    axes[row, col].set_ylabel("Avg. " + name)
plt.tight_layout()
plt.show()

### $\|f(x)\|$ by token type (Kobayashi et al., Section 4)
Kobayashi et al.'s explanation of the [SEP] result: the heads that put large
$\alpha$ on [SEP] have small $\|f(\text{[SEP]})\|$. Requires `fx_norms`
(`--outputs attns,norms,fx_norms`).

In [ ]:
if "fx_norms" in data[0]:
  groups = {"[CLS]": [], "[SEP]": [], ". or ,": [], "other": []}
  fx = {g: np.zeros((n_layers, n_heads)) for g in groups}
  counts = {g: 0 for g in groups}
  for doc in data:
    f = np.asarray(doc["fx_norms"], dtype=np.float64)
    for position, token in enumerate(doc["tokens"]):
      g = (token if token in ("[CLS]", "[SEP]") else
           ". or ," if token in (".", ",") else "other")
      fx[g] += f[:, :, position]
      counts[g] += 1
  plt.figure(figsize=(5, 4))
  for (g, total), color in zip(fx.items(), [RED, BLUE, PURPLE, GREY]):
    xs, ys, avgs = get_data_points(total / max(counts[g], 1))
    plt.scatter(xs, ys, s=8, color=color, label=g)
    plt.plot(1 + np.arange(n_layers), avgs, color=color)
  plt.xlabel("Layer")
  plt.ylabel("Avg. ||f(x)||")
  plt.legend()
  plt.show()
else:
  print("no fx_norms in the data")

### Entropies (Section 3.3)

In [ ]:
entropy_stats = {key: au.entropy_stats(data, key)
                 for key in ["attns", "norms"]}

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey=True)
for col, key in enumerate(["attns", "norms"]):
  uniform, entropies, entropies_cls = entropy_stats[key]
  for row, (values, label, color) in enumerate([
      (entropies, "heads", BLUE), (entropies_cls, "heads from [CLS]", RED)]):
    ax = axes[row, col]
    xs, es, avgs = get_data_points(values)
    ax.scatter(xs, es, c=color, s=5, label=label)
    ax.plot(1 + np.arange(n_layers), avgs, c=color)
    ax.plot([1, n_layers], [uniform, uniform], c="k", linestyle="--")
    ax.set_title(key + " (normalized rows)" if key != "attns" else key)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Avg. entropy (nats)")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Clustering heads (Section 6)
Uses the output of `head_distances.py` (JS divergences between heads; norms are
row-normalized first). Behaviour labels use Clark et al.'s thresholds, which were
set for attention; for norms they are applied to the *normalized* statistics.

In [ ]:
ENTROPY_THRESHOLD = 3.8
POSITION_THRESHOLD = 0.5
SPECIAL_TOKEN_THRESHOLD = 0.6
# heads identified in Clark et al. (for bert-base-uncased)
LINGUISTIC_HEADS = {(4, 3): "Coreference", (7, 10): "Determiner",
                    (7, 9): "Direct object", (8, 5): "Object of prep.",
                    (3, 9): "Passive auxiliary", (6, 5): "Possesive"}


def plot_clusters(key, stats, entropies, ax):
  js = utils.load_pickle(HEAD_DISTANCES[key])
  mds = sklearn.manifold.MDS(metric=True, n_init=5, eps=1e-10, max_iter=1000,
                             dissimilarity="precomputed", random_state=0)
  pts = mds.fit_transform(js).reshape((n_layers, n_heads, 2))
  seen = set()
  for layer in range(n_layers):
    for head in range(n_heads):
      label, color, marker, size = "", GREY, "o", 4
      if stats["right"][layer, head] > POSITION_THRESHOLD:
        label, color, marker = "attend to next", RED, ">"
      if stats["left"][layer, head] > POSITION_THRESHOLD:
        label, color, marker = "attend to prev.", BLUE, "<"
      if entropies[layer, head] > ENTROPY_THRESHOLD:
        label, color, marker = "attend broadly", ORANGE, "^"
      if stats["cls"][layer, head] > SPECIAL_TOKEN_THRESHOLD:
        label, color, marker, size = "attend to [CLS]", PURPLE, "$C$", 5
      if stats["sep"][layer, head] > SPECIAL_TOKEN_THRESHOLD:
        label, color, marker, size = "attend to [SEP]", GREEN, "$S$", 5
      if stats["punct"][layer, head] > SPECIAL_TOKEN_THRESHOLD:
        label, color, marker, size = "attend to . and ,", SEA, "s", 3.2
      x, y = pts[layer, head]
      if (layer, head) in LINGUISTIC_HEADS:
        label, color, marker = "", BLACK, "x"
        ax.text(x, y, LINGUISTIC_HEADS[(layer, head)], color=color)
      if label in seen:
        label = ""
      seen.add(label)
      ax.plot([x], [y], marker=marker, markersize=size, color=color,
              label=label, linestyle="")
  ax.set_xticks([])
  ax.set_yticks([])
  ax.legend(loc="best")
  ax.set_title(key)


if all(os.path.exists(p) for p in HEAD_DISTANCES.values()):
  fig, axes = plt.subplots(1, 2, figsize=(12, 6))
  plot_clusters("attns", avg_stats["attention"], entropy_stats["attns"][1],
                axes[0])
  plot_clusters("norms", avg_stats["norm (normalized)"],
                entropy_stats["norms"][1], axes[1])
  plt.show()
else:
  print("run head_distances.py first")